# Pattern 01: Naive dense retrieval

Follows this repo's mandatory 8-section notebook template -- this is one of "the 10 patterns."

## Reproducibility header

In [ ]:
import platform
import subprocess
import sys

import numpy
import openai

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


## Setup (loaded once, used by every section below)

In [ ]:
import os

os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import MockLLM, get_llm

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm

from recipes.embeddings import get_embedder

embedder = get_embedder()


## 1. What this pattern does

Naive dense retrieval embeds every corpus chunk once into a vector space (`text-embedding-3-small`
by default), embeds the incoming question the same way, and retrieves the top-k
chunks by vector distance (`recipes/store.py`'s sqlite-vec-backed `VectorStore`). It's the
"obvious" RAG baseline: no reranking, no query rewriting, no hybrid signal -- pure nearest-neighbor
search in embedding space.


## 2. When to use it

- A fast, simple baseline to establish before adding reranking, hybrid search, or query
  rewriting on top
- Queries and documents are semantically related but don't share exact vocabulary (dense
  embeddings generalize across paraphrase and synonymy where BM25 can't)
- You have an embedding API budget and don't need BM25's zero-cost retrieval


## 3. When NOT to use it

- Queries depend on exact identifiers, rare technical terms, model names, or numbers that dense
  embeddings don't represent distinctly (see section 7)
- You need retrieval explainability -- a cosine-similarity score is much harder to justify to a
  user than "this chunk shares 4 of your 5 query words"
- Cost/latency budget doesn't allow an embedding API call per query and per corpus chunk


## 4. Implementation

In [ ]:
from recipes.naive_dense import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

# Try it on one question directly.
sample = retrieve_and_answer("What does PCEval stand for?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("answer:", sample.answer)


## 5. Run on our eval set

In [ ]:
pattern_fn = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="01_naive_dense",
    judges_enabled=True,
)


## 6. Example query walkthrough

One example per eval-set category, showing the retrieved chunks and the final answer.

In [ ]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print()


## 7. Where this pattern FAILS

Computed on the full 66-question eval set with the local models behind the retrieval
leaderboard (`BAAI/bge-small-en-v1.5` embeddings, `BAAI/bge-reranker-v2-m3` reranker; see
`evals/retrieval_leaderboard.py --per-question` for every rank quoted here).

**Failure 1: right paper, wrong chunk.** Six of the 24 paraphrase questions miss the top 10
entirely, and in every one of them the top results are other chunks of the correct paper. For q22
(*what the task monitor does when the robot misses a time window*) the target is
`arxiv:2601.00087#15`; dense retrieval returns `#5`, then a PCEval chunk, then `#1` of the same
paper. For q52 (*whether abstention is the model's own opt-out or an external cutoff*) the target is
`arxiv:2601.00138#12` and the top three are `#9`, `#2`, `#8` of that paper. A paraphrase embeds to
the paper's topic cluster, and inside that cluster the chunk that states the mechanism is not
separated from its siblings. With 11 to 20 chunks per paper this is the dominant failure mode: the
pattern's MRR of 0.605 is the lowest on the leaderboard while its hit@10 is within a few points of
BM25's.

**Failure 2: exact numbers and named configurations.** Two keyword questions miss the top 10 where
BM25 ranks the target first. q29 asks how many nodes the mixed FCM had and how the two edge matrices
were combined (target `arxiv:2601.00097#8`); q61 asks for the draft-to-target execution-time ratio
on Llama3-70B versus Qwen3-8B (target `arxiv:2601.11580#14`). In both cases the top results are
neighbouring chunks of the same paper that discuss the same subject in general terms. Embeddings
carry topic, not specific values; a chunk that contains "24 nodes" or "12.5%" is not closer to a
question about those values than the chunk next to it.

**Failure 3: a question about two papers lands on one.** Both cross-paper multi-hop questions
(q31, q32) miss the top 10. For q32 (*what role GPT-4o plays in the tool-use rule-learning paper
and the PDE-discovery paper*) the top three are all from the PDE paper; the RIMRULE half of the
question is absent from the top 10. One query vector has one nearest region. Recall@10 across the
nine multi-hop questions is where this pattern loses most to the hybrid and reranked rows.

## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook.

```python
"""Minimal naive dense retrieval + generation, no eval harness."""
from recipes.embeddings import get_embedder
from recipes.naive_dense import make_retrieve_and_answer
from recipes.llm import get_llm

corpus_by_id = {}  # {chunk_id: {"text": ..., ...}, ...} -- fill in your own chunks
embedder = get_embedder()
llm = get_llm()

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, embedder=embedder, llm=llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer)
```
